In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import torch
from tqdm import tqdm

In [8]:
torch.device("mps")

device(type='mps')

In [9]:
dataset = pd.read_csv("ACT2_competition_training.csv")

thresh = 6.5 # from Bai et al.

total_Y = dataset['Act'].to_numpy()
total_X = dataset.drop(columns=['MOLECULE', 'Act']).to_numpy()

print(total_X.shape)

mdl = RandomForestRegressor(n_estimators=100, max_depth=20, max_features='sqrt')

(8716, 5877)


In [10]:
total_X = torch.tensor(total_X, dtype=torch.float32, device="mps")
tanimoto_num = total_X @ total_X.T
tanimoto_num = tanimoto_num.cpu().numpy()
total_X = total_X.cpu().numpy()

tanimoto_denom = (total_X**2).sum(axis=1) + (total_X**2).sum(axis=1)[:,np.newaxis] - tanimoto_num
tanimoto = tanimoto_num/tanimoto_denom # this is used here https://openreview.net/pdf?id=BbZVFj0QPv
np.save('similarityMatrix.npy', tanimoto)

In [11]:
def get_scores(Y, predictions, thresh):
  return np.where(Y > thresh, np.inf, thresh-predictions)

In [12]:
clf = RandomForestClassifier(n_estimators=100, max_depth=20, max_features='sqrt')


for seed in tqdm(range(250)):
    np.random.seed(seed)
    n_total = len(total_X)
    reind = np.random.permutation(n_total)

    # use the random permuation to create the split
    n_test = 400
    n_tc = n_total - n_test

    tc_idx = reind[:n_tc]
    test_idx = reind[n_tc:]

    n_train = n_tc - 400
    train_idx = tc_idx[:n_train]
    calib_idx = tc_idx[n_train:]

    Xtc = total_X[tc_idx]
    Ytc = total_Y[tc_idx]

    Xtest = total_X[test_idx]
    Ytest = total_Y[test_idx]

    Xtrain = total_X[train_idx]
    Ytrain = total_Y[train_idx]

    Xcalib = total_X[calib_idx]
    Ycalib = total_Y[calib_idx]

    mdl.fit(Xtrain, Ytrain)
    clf.fit(Xtrain, Ytrain <= thresh)

    combinedX = np.concatenate((Xcalib, Xtest))
    class_index = list(clf.classes_).index(True)
    log_likelihoods = clf.predict_log_proba(combinedX)[:, class_index]

    calib_pred, test_pred = mdl.predict(Xcalib), mdl.predict(Xtest)

    calibS, testS = get_scores(Ycalib, calib_pred, thresh), thresh-test_pred

    final_perm = np.concatenate((calib_idx, test_idx))
    np.save(f'./permutations/perm_j{seed}.npy', final_perm)
    
    adjusted_testY = Ytest - thresh
    np.save(f'./scores_and_Ys/calibS_j{seed}.npy', calibS)
    np.save(f'./scores_and_Ys/testS_j{seed}.npy', testS)
    np.save(f'./scores_and_Ys/testY_j{seed}.npy', adjusted_testY)

    np.save(f'./log_likelihoods/log_likelihoods_j{seed}.npy', log_likelihoods)

100%|██████████| 250/250 [43:42<00:00, 10.49s/it]


In [13]:
# get avg value of 2/lambda_max to choose gamma values.

inv_e_values = []
for seed in tqdm(range(250)):
    np.random.seed(seed)
    n_total = len(total_X)
    reind = np.random.permutation(n_total)

    # use the random permuation to create the split
    n_test = 400
    n_tc = n_total - n_test

    tc_idx = reind[:n_tc]
    test_idx = reind[n_tc:]

    n_train = n_tc - 400
    train_idx = tc_idx[:n_train]
    calib_idx = tc_idx[n_train:]

    final_perm = np.concatenate((calib_idx, test_idx))


    similarityMatrix_for_trial = tanimoto[final_perm][:,final_perm]
    inv_e_val = 2./(np.linalg.eigvals(similarityMatrix_for_trial).max())
    inv_e_values.append(inv_e_val)

100%|██████████| 250/250 [01:32<00:00,  2.69it/s]


In [ ]:
np.mean(inv_e_values)